# [SOLUTION] Udaplay Project

## Part 02 - AI Agent Development

Build an intelligent agent that combines local knowledge (RAG) with web search capabilities.

**Agent Name:** UdaPlay — AI Research Agent for the video game industry.

The agent will:
1. Answer questions using internal knowledge (vector DB)
2. Evaluate the quality of retrieved results
3. Search the web when internal knowledge is insufficient
4. Maintain conversation state across queries

---
### Setup

In [ ]:
# Only needed for Udacity workspace — workaround for pysqlite3
import importlib.util
import sys

if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [ ]:
import os
import json
from typing import Optional

import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [ ]:
# Load environment variables from config.env
load_dotenv("config.env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://openai.vocareum.com/v1")

print(f"OPENAI_API_KEY set: {OPENAI_API_KEY is not None}")
print(f"TAVILY_API_KEY set: {TAVILY_API_KEY is not None}")

---
### Load ChromaDB Collection

Reload the persistent vector database created in Part 1.

In [ ]:
# Load the persistent ChromaDB client and collection
CHROMA_DB_PATH = "chromadb_udaplay"
COLLECTION_NAME = "udaplay"

chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    api_base=OPENAI_BASE_URL
)

collection = chroma_client.get_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn
)

count = collection.count()
print(f"Loaded collection '{COLLECTION_NAME}' with {count} documents!")

---
### Define Tools

Build at least 3 tools:
1. **retrieve_game** — Search the vector DB for game information
2. **evaluate_retrieval** — Assess whether retrieved documents sufficiently answer the query
3. **game_web_search** — Fall back to web search when internal knowledge is insufficient

#### Tool 1: retrieve_game

Semantic search over the local vector database of game information.

In [ ]:
retrieve_game = tool(
    name="retrieve_game",
    description=(
        "Semantic search: Finds most relevant game entries in the vector database. "
        "Returns results with Name, Platform, YearOfRelease, Genre, Publisher, and Description. "
        "Use this FIRST to answer any game-related question."
    )
)(
    lambda query: _retrieve_game_impl(query)
)

def _retrieve_game_impl(query: str) -> str:
    """Query the ChromaDB vector store for game information."""
    results = collection.query(
        query_texts=[query],
        n_results=5,
        include=['documents', 'distances', 'metadatas']
    )

    if not results['documents'][0]:
        return json.dumps({"found": False, "message": "No results found for this query."})

    output = {"found": True, "results": []}
    for meta, doc, dist in zip(
        results['metadatas'][0],
        results['documents'][0],
        results['distances'][0]
    ):
        output["results"].append({
            "name": meta.get("Name"),
            "platform": meta.get("Platform"),
            "year": meta.get("YearOfRelease"),
            "publisher": meta.get("Publisher"),
            "genre": meta.get("Genre"),
            "description": meta.get("Description"),
            "similarity_score": round(1 - dist, 4),
            "content": doc
        })

    return json.dumps(output, indent=2)

#### Tool 2: evaluate_retrieval

Uses an LLM judge to assess whether the retrieved documents are sufficient to answer the user's question.

In [ ]:
class EvaluationReport(BaseModel):
    """Structured output for retrieval evaluation."""
    useful: bool = Field(description="Whether the retrieved documents are useful to answer the question")
    confidence: float = Field(
        description="Confidence score between 0.0 and 1.0",
        ge=0.0, le=1.0
    )
    explanation: str = Field(description="Detailed explanation of the evaluation result")


# Create an LLM instance for evaluation (separate from the agent's LLM)
eval_llm = LLM(model="gpt-4o-mini", temperature=0.0)


def _evaluate_retrieval_impl(question: str, retrieved_docs: str) -> str:
    """Evaluate whether retrieved documents are sufficient to answer the question."""
    prompt = (
        "Your task is to evaluate if the following retrieved documents are enough "
        "to respond to the user's question.\n\n"
        f"Question: {question}\n\n"
        f"Retrieved Documents:\n{retrieved_docs}\n\n"
        "Evaluate whether:"
        " 1. The documents contain the specific information needed to answer the question.\n"
        " 2. The information is complete enough.\n"
        " 3. If not, what specific information is missing.\n\n"
        "Give a detailed explanation so it's possible to take action."
    )

    response = eval_llm.invoke(prompt, response_format=EvaluationReport)

    try:
        report = EvaluationReport.model_validate_json(response.content)
    except Exception:
        # Fallback: parse as best we can
        report = EvaluationReport(
            useful=True,
            confidence=0.5,
            explanation='Parsing error - defaulting to "useful".'
        )

    return json.dumps(report.model_dump(), indent=2)


evaluate_retrieval = tool(
    name="evaluate_retrieval",
    description=(
        "Based on the user's question and the list of retrieved documents, "
        "analyze whether the documents are useful to answer the question. "
        "Returns: useful (bool), confidence (0-1), and explanation."
    )
)(
    lambda question, retrieved_docs: _evaluate_retrieval_impl(question, retrieved_docs)
)

#### Tool 3: game_web_search

Use the Tavily API to search the web when the vector database doesn't have sufficient information.

In [ ]:
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)


def _game_web_search_impl(question: str) -> str:
    """Search the web for game industry information using Tavily."""
    try:
        response = tavily_client.search(
            query=question,
            search_depth="advanced",
            max_results=5
        )

        results = response.get("results", [])
        if not results:
            return json.dumps({"found": False, "message": "No web results found."})

        output = {"found": True, "results": []}
        for r in results:
            output["results"].append({
                "title": r.get("title"),
                "content": r.get("content"),
                "url": r.get("url")
            })

        return json.dumps(output, indent=2)

    except Exception as e:
        return json.dumps({
            "found": False,
            "error": f"Web search failed: {str(e)}"
        })


game_web_search = tool(
    name="game_web_search",
    description=(
        "Search the web for game industry information. "
        "Use this when the local vector database does not have enough information "
        "to answer the user's question. Returns title, content snippets, and URLs."
    )
)(
    lambda question: _game_web_search_impl(question)
)

---
### Create the Agent

Equip the agent with all three tools and a well-crafted system prompt.

In [ ]:
# System instructions for the UdaPlay agent
AGENT_INSTRUCTIONS = (
    "You are UdaPlay, an AI Research Agent specializing in the video game industry. "
    "Your goal is to answer user questions about video games accurately and thoroughly.\n\n"
    "WORKFLOW:\n"
    "1. FIRST, use the `retrieve_game` tool to search your internal vector database "
    "for relevant game information.\n"
    "2. Use `evaluate_retrieval` to assess whether the retrieved documents are sufficient "
    "to answer the user's question.\n"
    "3. If the documents ARE sufficient (useful=true, confidence>=0.6), "
    "answer using that information.\n"
    "4. If the documents are NOT sufficient, use `game_web_search` to find "
    "additional information from the web.\n"
    "5. Combine information from multiple sources when needed.\n\n"
    "OUTPUT REQUIREMENTS:\n"
    "- Always cite your information sources (which games from the database, or which URLs).\n"
    "- Present information in a natural, readable format.\n"
    "- If you cannot find an answer, clearly state that and suggest what the user could try.\n"
    "- Provide confidence levels in your answers.\n"
    "- Keep answers concise but informative."
)

# Create the agent with all three tools
agent = Agent(
    model_name="gpt-4o-mini",
    instructions=AGENT_INSTRUCTIONS,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.3
)

print("Agent 'UdaPlay' created successfully!")
print(f"Tools available: {[t.name for t in agent.tools]}")

---
### Test the Agent

Let's test the agent with three example queries.

#### Query 1: Pokémon release date

This query should be answerable from the local vector database (game #006 — Pokémon Gold and Silver).

In [ ]:
run_1 = agent.invoke("When was Pokémon Gold and Silver released?")
final_state_1 = run_1.get_final_state()
print("=" * 70)
print("QUERY 1: When was Pokémon Gold and Silver released?")
print("=" * 70)

# Extract and display the final answer
if final_state_1:
    messages = final_state_1.get("messages", [])
    for msg in reversed(messages):
        if isinstance(msg, AIMessage) and msg.content and not msg.tool_calls:
            print(f"\n{msg.content}")
            break
    print(f"\n--- Total tokens used: {final_state_1.get('total_tokens', 'N/A')} ---")

#### Query 2: First 3D Mario platformer

This should retrieve Super Mario 64 (game #009) — the first 3D Mario platformer.

In [ ]:
run_2 = agent.invoke("Which one was the first 3D platformer Mario game?")
final_state_2 = run_2.get_final_state()
print("=" * 70)
print("QUERY 2: Which one was the first 3D platformer Mario game?")
print("=" * 70)

if final_state_2:
    messages = final_state_2.get("messages", [])
    for msg in reversed(messages):
        if isinstance(msg, AIMessage) and msg.content and not msg.tool_calls:
            print(f"\n{msg.content}")
            break
    print(f"\n--- Total tokens used: {final_state_2.get('total_tokens', 'N/A')} ---")

#### Query 3: Mortal Kombat X on PS5

This query tests the agent's ability to handle negative information. Mortal Kombat X was released for PS4, not PS5. The local DB may not have this data, so the agent should fall back to web search.

In [ ]:
run_3 = agent.invoke("Was Mortal Kombat X released for PlayStation 5?")
final_state_3 = run_3.get_final_state()
print("=" * 70)
print("QUERY 3: Was Mortal Kombat X released for PlayStation 5?")
print("=" * 70)

if final_state_3:
    messages = final_state_3.get("messages", [])
    for msg in reversed(messages):
        if isinstance(msg, AIMessage) and msg.content and not msg.tool_calls:
            print(f"\n{msg.content}")
            break
    print(f"\n--- Total tokens used: {final_state_3.get('total_tokens', 'N/A')} ---")

---
### Session Memory Demo

Show that the agent maintains conversation state across queries within the same session.

In [ ]:
# Ask a follow-up question in the same session
run_followup = agent.invoke("What about Pokémon Ruby and Sapphire?", session_id="default")
final_state_followup = run_followup.get_final_state()

print("=" * 70)
print("FOLLOW-UP: What about Pokémon Ruby and Sapphire?")
print("(maintains session context from previous queries)")
print("=" * 70)

if final_state_followup:
    messages = final_state_followup.get("messages", [])
    for msg in reversed(messages):
        if isinstance(msg, AIMessage) and msg.content and not msg.tool_calls:
            print(f"\n{msg.content}")
            break
    print(f"\n--- Total tokens used: {final_state_followup.get('total_tokens', 'N/A')} ---")

---
### Summary

**What was built:**

| Component | Status |
|---|---|
| ✅ `retrieve_game` tool | Semantic search over the vector DB |
| ✅ `evaluate_retrieval` tool | LLM judge assessing answer quality |
| ✅ `game_web_search` tool | Web search fallback via Tavily |
| ✅ Agent with State Machine | Workflow: retrieve → evaluate → (answer | web search) |
| ✅ Conversation Memory | Maintains context across queries in a session |
| ✅ Structured Output | Clean, cited answers with confidence |

**Architecture:**
```
User Query
    │
    ▼
┌─────────────────┐
│  retrieve_game   │ ◄── ChromaDB (15 games)
└────────┬────────┘
         │
         ▼
┌─────────────────────┐
│ evaluate_retrieval  │ ◄── LLM Judge (gpt-4o-mini)
└────────┬────────────┘
         │
    ┌────┴────┐
    ▼         ▼
  [Answer]  [Web Search] ──► Tavily API
```